# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  fps_model                linear
  window                   120
  target_delay             20046
  initial_time             2000-07-24T14:45:55
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           19340 frames  (322.33s)


In [16]:
x.adjust(strategy_name="six-bait-then-balls")

[expedition] strategy_name = 'six-bait-then-balls'


# Chart - Finding a target

## Create chart

In [2]:
x.precompute_chart()
x.save()

[expedition] 15:27:09  === precompute_chart ===  (2026-09-10)
[expedition] 15:27:09  charting metang key_seed=0x0C0E02C2 delay=180-600s strategy=six-bait-then-balls criteria=machete-50-turns-after-5-balls  fps_models=['linear', 'quad'] (union)  workers=12
[expedition] 15:27:10  mdmsh 1/248  elapsed 0.0m  eta ~7.6m
[expedition] 15:27:10  mdmsh 5/248  elapsed 0.0m  eta ~1.5m
[expedition] 15:27:10  mdmsh 10/248  elapsed 0.0m  eta ~0.7m
[expedition] 15:27:10  mdmsh 15/248  elapsed 0.0m  eta ~0.5m
[expedition] 15:27:10  mdmsh 20/248  elapsed 0.0m  eta ~0.4m
[expedition] 15:27:10  mdmsh 25/248  elapsed 0.0m  eta ~0.3m
[expedition] 15:27:10  mdmsh 30/248  elapsed 0.0m  eta ~0.2m
[expedition] 15:27:10  mdmsh 35/248  elapsed 0.0m  eta ~0.2m
[expedition] 15:27:10  mdmsh 40/248  elapsed 0.0m  eta ~0.2m
[expedition] 15:27:10  mdmsh 45/248  elapsed 0.0m  eta ~0.1m
[expedition] 15:27:10  mdmsh 50/248  elapsed 0.0m  eta ~0.1m
[expedition] 15:27:10  mdmsh 55/248  elapsed 0.0m  eta ~0.1m
[expedition] 1

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [7]:
x.chart_report()
x.save()

[expedition] 15:53:12  === chart_report ===
Best (boot time, M) pairs  [top 10 of 9]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-07-27 14:53:26     268203       16486     273       27.9%    62.1
   2  2000-06-30 14:50:38     195202       12110     200       27.6%    52.9
   3  2000-05-30 14:59:59     401460       24474     407       27.5%    75.9
   4  2000-08-26 14:57:03     411536       25078     417       27.3%    76.9
   5  2000-07-29 14:57:08     463584       28198     469       26.9%    81.6
   6  2000-06-28 14:51:49     303502       18602     309       26.9%    66.0
   7  2000-07-28 14:59:13     279547       17166     285       26.7%    63.4
   8  2000-06-27 14:51:55     180055       11202     185       26.5%    50.8
   9  2000-06-27 14:47:59     221226       13670     227       26.5%    56.4
[expedition] 15:53:42  best target for each of 2464 starting times also saved
[expedition] 15:53:42  findings saved -> d

## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [8]:
x.select_target()

[expedition] 15:57:51  === select_target ===



Select by  [t] top ranking   [s] specific starting time   [l] reuse last (07-24 14:45:55)  (blank to cancel):  l


  -> best target for 2000-07-24 14:45:55: M=180055 ms, F_b=11202, P~26.5%
[expedition] Saved to data/expeditions/metang.json
[expedition] 15:57:52  target set: boot 2000-07-24T14:45:55, timer M=180055 ms, expected F_b=11202 (P~26.5%). Saved.
[expedition] 15:57:52  predicted battle time (m/d h:m:s): 07-24 14:49:00  (= boot + 185s; year is the chart's 2000)


{'rank': 2125,
 'initial_time': '2000-07-24T14:45:55',
 'M': 180055,
 'target_delay': 11202,
 'second': 185,
 'p': 0.2652531709356521,
 'sigma': 50.847634332688095,
 'mdmsh': [217, 14]}

## Examine target area

In [9]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=180055 ms  ->  mean F_b=11202.0 (target_delay=11202)  sigma=50.8
RTC-second distribution (σ_S=0.52s):  185=57%  186=36%  184=5%  187=1%

=== second 185  P(S=185)=57.0%   mdmsh(m,h)=(217, 14)   battle 07-24 14:49:00
    cp(this second) = 27.91%   ->  contributes P·cp = 15.92% to the total
      frame      Δ        seed  hit    weight        w%      cumP%
    --------------------------------------------------------------
      11024   -178  0xD90E2B10    ✗    0.0022    0.002%     0.000%
      11025   -177  0xD90E2B11    ✗    0.0023    0.002%     0.000%
      11026   -176  0xD90E2B12    ✗    0.0025    0.002%     0.000%
      11027   -175  0xD90E2B13    ✗    0.0027    0.002%     0.000%
      11028   -174  0xD90E2B14    ✗    0.0029    0.002%     0.000%
      11029   -173  0xD90E2B15    ✗    0.0031    0.002%     0.000%
      11030   -172  0xD90E2B16    ✗    0.0033    0.003%     0.000%
      11031   -171  0x

{'p': 0.2652486713567191,
 'seconds': [{'second': 185,
   'p_second': 0.5703108096529135,
   'cp': 0.2790973305872824},
  {'second': 186, 'p_second': 0.36488422385945946, 'cp': 0.2512166061501245},
  {'second': 184, 'p_second': 0.05245433762006336, 'cp': 0.22158780751618048},
  {'second': 187,
   'p_second': 0.012350628867563775,
   'cp': 0.22575601483766392}],
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [ ]:
x.metronome_compass()
x.save()

## Finding what seed you hit in safari

`compass_safari()` builds candidates from the calibrated model: for the commanded countdown **M** (set by `select_target`) it sweeps the battle-frame window **F\* ± kσ** across second offsets **δ∈{−1,0,+1}** (off-by-one timer-start timing — each δ uses the *same* frame window). No hand-set delay window.

As you enter observed turns it ranks survivors by **posterior landing probability** (`P(land)`), shows the most-likely seed and which **δ** you hit ("timer on time / +1s late"), and flags when one candidate passes the confidence threshold. Extra commands:

- **`w`** — widen the frame (`k`) and/or second (`±K`) window and re-apply your path so far (also offered automatically on a no-match).
- The set is bounded to the seeds carrying `mass_cap` (default 0.999) of the landing probability; the Jane offload tip triggers on the *prior-weighted* effective count.

Pass `second_offsets=` / `mass_cap=` to override. Afterwards, `x.save_safari_run()` logs the identified seed, observed path, and inferred timer offset to `data/safari_runs.jsonl` (no capture required) for future model retuning.

In [ ]:
x.compass_safari()
x.save()

In [ ]:
# Loop-back: log this run (seed, observed path, inferred timer offset) for model retuning.
# No capture required — records even a fled/ambiguous run.
x.save_safari_run()

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()